# Извлечение DINOv2 эмбеддингов фото

Прогоняет 144313 фото объявлений через замороженный DINOv2-S и сохраняет эмбеддинги в parquet

Вход: zip с папкой image, загружен как Kaggle Dataset
Выход: /kaggle/working/image_emb_dinov2.parquet (image_id + 384 признака)

Запуск: Accelerator GPU T4 x2, Internet ON, Run All

In [ ]:
!pip install -q "transformers>=4.38"

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import AutoModel

In [ ]:
MODEL_NAME = 'facebook/dinov2-small'
IMG_SIZE = 224
RESIZE_SIZE = 256
BATCH_SIZE = 128
NUM_WORKERS = 4
EMB_DIM = 384
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
INPUT_DIR = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working')
# папка с фото внутри датасета, поменяй если имя датасета другое, None = искать автоматически
IMAGES_ROOT = '/kaggle/input/avito-film-camera-body-type/image'
OUT_PARQUET = WORK_DIR / 'image_emb_dinov2.parquet'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EXPECTED_ROWS = 144313

## Шаг 1: найти фото

In [ ]:
def find_images_root():
    """вернуть Path к папке с фото: заданную руками, готовый датасет или распакованный zip"""
    if IMAGES_ROOT is not None and Path(IMAGES_ROOT).exists():
        return Path(IMAGES_ROOT)
    if next(INPUT_DIR.rglob('*.jpg'), None) is not None:
        return INPUT_DIR
    zip_path = next(INPUT_DIR.rglob('*.zip'))
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(WORK_DIR / 'images')
    return WORK_DIR / 'images'

In [ ]:
images_root = find_images_root()
print('фото лежат в', images_root)

## Шаг 2: список путей

In [ ]:
def list_image_paths(root):
    """все пути к jpg отсортированные по image_id"""
    paths = list(Path(root).rglob('*.jpg'))
    return sorted(paths, key=lambda p: int(p.stem))

In [ ]:
paths = list_image_paths(images_root)
print('найдено фото:', len(paths))

## Шаг 3: датасет и препроцессинг

In [ ]:
def build_transform():
    """препроцессинг под DINOv2"""
    return transforms.Compose([
        transforms.Resize(RESIZE_SIZE),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

In [ ]:
class ImageFiles(Dataset):
    """отдает image_id и тензор картинки"""

    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        image = Image.open(self.paths[i]).convert('RGB')
        return int(self.paths[i].stem), self.transform(image)

In [ ]:
def build_loader(paths):
    """собрать даталоадер по списку путей"""
    dataset = ImageFiles(paths, build_transform())
    return DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)

## Шаг 4: модель

In [ ]:
def load_model():
    """загрузить DINOv2 на gpu в режиме оценки"""
    model = AutoModel.from_pretrained(MODEL_NAME)
    model.eval().to(DEVICE)
    if torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)
    return model

In [ ]:
def embed_batch(model, pixels):
    """эмбеддинги одного батча как float32 numpy"""
    pixels = pixels.to(DEVICE, non_blocking=True)
    out = model(pixel_values=pixels)
    return out.pooler_output.float().cpu().numpy()

## Шаг 5: прогон

In [ ]:
@torch.no_grad()
def extract_all(model, loader):
    """прогнать все батчи и собрать id и эмбеддинги"""
    ids, embs = [], []
    for batch_ids, pixels in tqdm(loader):
        embs.append(embed_batch(model, pixels))
        ids.append(batch_ids.numpy())
    return np.concatenate(ids), np.concatenate(embs)

In [ ]:
model = load_model()
ids, embs = extract_all(model, build_loader(paths))
print('эмбеддинги:', embs.shape)

## Шаг 6: собрать и сохранить

In [ ]:
def to_dataframe(ids, embs):
    """собрать таблицу image_id плюс столбцы признаков"""
    columns = [f'emb_{i}' for i in range(EMB_DIM)]
    df = pd.DataFrame(embs.astype('float32'), columns=columns)
    df.insert(0, 'image_id', ids.astype('int64'))
    return df

In [ ]:
def save_parquet(df, path):
    """сохранить таблицу в parquet"""
    df.to_parquet(path, index=False)
    return path

In [ ]:
emb_df = to_dataframe(ids, embs)
save_parquet(emb_df, OUT_PARQUET)
print('сохранено в', OUT_PARQUET)

## Шаг 7: проверки

In [ ]:
assert len(emb_df) == EXPECTED_ROWS, f'строк {len(emb_df)} вместо {EXPECTED_ROWS}'
assert emb_df['image_id'].nunique() == EXPECTED_ROWS, 'image_id не уникален'
assert emb_df.isna().sum().sum() == 0, 'есть NaN'
assert emb_df.shape[1] == EMB_DIM + 1, 'неверное число столбцов'
print('строк:', len(emb_df))
print('признаков:', EMB_DIM)
print('OK')

## Что скачать

Скачай image_emb_dinov2.parquet из панели Output справа

Локально положи в training/artifacts/embeddings/image_emb_dinov2.parquet - это вход для этапа 3